# Lab 2 — Dimensional Data Warehouse
## Modelado Dimensional: Retail Technology Sales

> **ETL (G01) — Universidad EAFIT**
> Unit 1, Activity 3

---

### Objetivo del Notebook

Este notebook presenta el **modelado dimensional** del Data Warehouse, incluyendo:

1. Diseño del Star Schema
2. Estructura de dimensiones y tabla de hechos
3. Carga de datos desde fuentes CSV/JSON
4. Validación de integridad referencial
5. Queries analíticas (R1-R5)
6. Visualizaciones de negocio

---
## 1. Diseño del Star Schema

### Flujo del Pipeline ETL

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   EXTRACT    │───►│  TRANSFORM   │───►│     LOAD     │───►│    QUERY     │
│              │    │              │    │              │    │              │
│ • CSV/JSON   │    │ • Map IDs    │    │ • Dimensions │    │ • R1-R5      │
│ • Validate   │    │ • Calc KPIs  │    │ • Fact Table │    │ • KPIs       │
└──────────────┘    │ • Surrogate  │    │ • FK constr. │    │ • Dashboards │
                    │   Keys       │    └──────────────┘    └──────────────┘
                    └──────────────┘
```

### Star Schema Diagram

```
                           ┌──────────────────────┐
                           │      DimDate          │
                           ├──────────────────────┤
                           │ ● date_key (PK)      │
                           │   full_date           │
                           │   day / month / year  │
                           │   month_name          │
                           └──────────┬───────────┘
                                      │
┌────────────────────┐     ┌──────────┴──────────┐     ┌────────────────────┐
│    DimProduct      │     │     FactSales        │     │     DimStore       │
├────────────────────┤     ├─────────────────────┤     ├────────────────────┤
│ ● product_key (PK) │◄────│ product_key (FK)    │────►│ ● store_key (PK)  │
│   product_id       │     │ date_key (FK)       │     │   store_id        │
│   product_name     │     │ store_key (FK)      │     │   store_name      │
│   category         │     │ channel_key (FK)    │     │   city            │
│   brand            │     │ promotion_key (FK)  │     │   region          │
│   list_price       │     │                     │     │   channel_id      │
│   unit_cost        │     │ ─── Measures ───    │     └────────────────────┘
└────────────────────┘     │ quantity            │
                           │ gross_sales         │
┌────────────────────┐     │ net_sales           │     ┌────────────────────┐
│    DimChannel      │     │ discount_amount     │     │   DimPromotion     │
├────────────────────┤     │ cost_amount         │     ├────────────────────┤
│ ● channel_key (PK) │◄────│ channel_key (FK)    │────►│ ● promotion_key    │
│   channel_id       │     │ gross_profit        │     │   promotion_id     │
│   channel_name     │     └─────────────────────┘     │   promotion_name   │
└────────────────────┘                                  │   discount_pct     │
                                                       └────────────────────┘
```

---
## 2. Configuración del Entorno

In [ ]:
import sqlite3
import json
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Markdown

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DB_PATH = os.path.join(ROOT, 'database', 'retail_dw.db')
DATA_DIR = os.path.join(ROOT, 'data')

print(f"Project root: {ROOT}")
print(f"Database:     {DB_PATH}")
print(f"Data dir:     {DATA_DIR}")

---
## 3. Crear Schema del Data Warehouse

In [ ]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS DimDate (
    date_key    INTEGER PRIMARY KEY,
    full_date   TEXT NOT NULL,
    day         INTEGER,
    month       INTEGER,
    year        INTEGER,
    month_name  TEXT
);

CREATE TABLE IF NOT EXISTS DimProduct (
    product_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id   TEXT NOT NULL,
    product_name TEXT,
    category     TEXT,
    brand        TEXT,
    list_price   REAL,
    unit_cost    REAL
);

CREATE TABLE IF NOT EXISTS DimStore (
    store_key   INTEGER PRIMARY KEY AUTOINCREMENT,
    store_id    TEXT NOT NULL,
    store_name  TEXT,
    city        TEXT,
    region      TEXT,
    channel_id  TEXT
);

CREATE TABLE IF NOT EXISTS DimChannel (
    channel_key  INTEGER PRIMARY KEY AUTOINCREMENT,
    channel_id   TEXT NOT NULL,
    channel_name TEXT
);

CREATE TABLE IF NOT EXISTS DimPromotion (
    promotion_key   INTEGER PRIMARY KEY AUTOINCREMENT,
    promotion_id    TEXT NOT NULL,
    promotion_name  TEXT,
    discount_pct    REAL
);

CREATE TABLE IF NOT EXISTS FactSales (
    sale_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key         INTEGER NOT NULL,
    product_key      INTEGER NOT NULL,
    store_key        INTEGER NOT NULL,
    channel_key      INTEGER NOT NULL,
    promotion_key    INTEGER NOT NULL,
    quantity         INTEGER,
    gross_sales      REAL,
    net_sales        REAL,
    discount_amount  REAL,
    cost_amount      REAL,
    gross_profit     REAL,
    FOREIGN KEY (date_key)      REFERENCES DimDate(date_key),
    FOREIGN KEY (product_key)   REFERENCES DimProduct(product_key),
    FOREIGN KEY (store_key)     REFERENCES DimStore(store_key),
    FOREIGN KEY (channel_key)   REFERENCES DimChannel(channel_key),
    FOREIGN KEY (promotion_key) REFERENCES DimPromotion(promotion_key)
);
"""

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA_SQL)
conn.commit()

print("[OK] Schema created")
print(f"Database: {DB_PATH}")

---
## 4. Cargar Dimensiones

In [ ]:
def load_dim_date(conn):
    df = pd.read_csv(os.path.join(DATA_DIR, 'sales_transactions.csv'))
    dates = pd.to_datetime(df['sale_date']).dt.date.unique()
    cursor = conn.cursor()
    for d in sorted(dates):
        cursor.execute(
            "INSERT OR IGNORE INTO DimDate (date_key, full_date, day, month, year, month_name) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (int(d.strftime('%Y%m%d')), str(d), d.day, d.month, d.year, d.strftime('%B'))
        )
    conn.commit()
    print(f"[OK] DimDate: {len(dates)} dates")


def load_dim_product(conn):
    with open(os.path.join(DATA_DIR, 'reference_data.json'), 'r') as f:
        ref = json.load(f)
    cursor = conn.cursor()
    for p in ref['products']:
        cursor.execute(
            "INSERT INTO DimProduct (product_id, product_name, category, brand, list_price, unit_cost) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (p['product_id'], p['product_name'], p['category'], p['brand'], p['list_price'], p['unit_cost'])
        )
    conn.commit()
    print(f"[OK] DimProduct: {len(ref['products'])} products")


def load_dim_store(conn):
    with open(os.path.join(DATA_DIR, 'reference_data.json'), 'r') as f:
        ref = json.load(f)
    cursor = conn.cursor()
    for s in ref['stores']:
        cursor.execute(
            "INSERT INTO DimStore (store_id, store_name, city, region, channel_id) "
            "VALUES (?, ?, ?, ?, ?)",
            (s['store_id'], s['store_name'], s['city'], s['region'], s['channel_id'])
        )
    conn.commit()
    print(f"[OK] DimStore: {len(ref['stores'])} stores")


def load_dim_channel(conn):
    with open(os.path.join(DATA_DIR, 'reference_data.json'), 'r') as f:
        ref = json.load(f)
    cursor = conn.cursor()
    for c in ref['channels']:
        cursor.execute(
            "INSERT INTO DimChannel (channel_id, channel_name) VALUES (?, ?)",
            (c['channel_id'], c['channel_name'])
        )
    conn.commit()
    print(f"[OK] DimChannel: {len(ref['channels'])} channels")


def load_dim_promotion(conn):
    with open(os.path.join(DATA_DIR, 'reference_data.json'), 'r') as f:
        ref = json.load(f)
    cursor = conn.cursor()
    for pr in ref['promotions']:
        cursor.execute(
            "INSERT INTO DimPromotion (promotion_id, promotion_name, discount_pct) VALUES (?, ?, ?)",
            (pr['promotion_id'], pr['promotion_name'], pr['discount_pct'])
        )
    conn.commit()
    print(f"[OK] DimPromotion: {len(ref['promotions'])} promotions")


load_dim_date(conn)
load_dim_product(conn)
load_dim_store(conn)
load_dim_channel(conn)
load_dim_promotion(conn)

---
## 5. Cargar Tabla de Hechos (FactSales)

In [ ]:
def get_key_mapping(conn, table, id_col, key_col):
    cursor = conn.cursor()
    cursor.execute(f"SELECT {id_col}, {key_col} FROM {table}")
    return {row[0]: row[1] for row in cursor.fetchall()}


def load_fact_sales(conn):
    cursor = conn.cursor()

    date_map = get_key_mapping(conn, 'DimDate', 'full_date', 'date_key')
    product_map = get_key_mapping(conn, 'DimProduct', 'product_id', 'product_key')
    store_map = get_key_mapping(conn, 'DimStore', 'store_id', 'store_key')
    channel_map = get_key_mapping(conn, 'DimChannel', 'channel_id', 'channel_key')
    promotion_map = get_key_mapping(conn, 'DimPromotion', 'promotion_id', 'promotion_key')

    cursor.execute("SELECT product_id, list_price, unit_cost FROM DimProduct")
    product_prices = {row[0]: (row[1], row[2]) for row in cursor.fetchall()}

    sales_df = pd.read_csv(os.path.join(DATA_DIR, 'sales_transactions.csv'))

    count = 0
    for _, row in sales_df.iterrows():
        date_key = int(pd.to_datetime(row['sale_date']).strftime('%Y%m%d'))
        product_key = product_map.get(row['product_id'])
        store_key = store_map.get(row['store_id'])
        channel_key = channel_map.get(row['channel_id'])
        promotion_key = promotion_map.get(row['promotion_id'])

        if None in (product_key, store_key, channel_key, promotion_key):
            continue

        quantity = row['quantity']
        unit_price_sale = row['unit_price_sale']
        list_price, unit_cost = product_prices[row['product_id']]

        gross_sales = quantity * list_price
        net_sales = quantity * unit_price_sale
        discount_amount = gross_sales - net_sales
        cost_amount = quantity * unit_cost
        gross_profit = net_sales - cost_amount

        cursor.execute(
            "INSERT INTO FactSales "
            "(date_key, product_key, store_key, channel_key, promotion_key, "
            "quantity, gross_sales, net_sales, discount_amount, cost_amount, gross_profit) "
            "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (date_key, product_key, store_key, channel_key, promotion_key,
             quantity, gross_sales, net_sales, discount_amount, cost_amount, gross_profit)
        )
        count += 1

    conn.commit()
    print(f"[OK] FactSales: {count} rows loaded")


load_fact_sales(conn)

---
## 6. Validación del Modelo Dimensional

In [ ]:
tables = {
    'DimDate': 'SELECT COUNT(*) FROM DimDate',
    'DimProduct': 'SELECT COUNT(*) FROM DimProduct',
    'DimStore': 'SELECT COUNT(*) FROM DimStore',
    'DimChannel': 'SELECT COUNT(*) FROM DimChannel',
    'DimPromotion': 'SELECT COUNT(*) FROM DimPromotion',
    'FactSales': 'SELECT COUNT(*) FROM FactSales',
}

print("=" * 50)
print("  Tabla              | Registros")
print("=" * 50)
for name, sql in tables.items():
    count = conn.execute(sql).fetchone()[0]
    print(f"  {name:<20} | {count:>8}")
print("=" * 50)

---
## 7. Muestra de Dimensiones

In [ ]:
dims = ['DimDate', 'DimProduct', 'DimStore', 'DimChannel', 'DimPromotion']

for dim in dims:
    display(Markdown(f"### {dim}"))
    df = pd.read_sql_query(f"SELECT * FROM {dim} LIMIT 10", conn)
    display(df)
    print()

In [ ]:
display(Markdown("### FactSales (primeras 10 filas)"))
df_fact = pd.read_sql_query("SELECT * FROM FactSales LIMIT 10", conn)
display(df_fact)

---
## 8. Queries Analíticas (R1-R5)

In [ ]:
QUERIES = {
    "R1 - Monthly Net Sales Trend": """
        SELECT d.year, d.month, d.month_name,
               SUM(f.net_sales) AS total_net_sales
        FROM FactSales f
        JOIN DimDate d ON f.date_key = d.date_key
        GROUP BY d.year, d.month, d.month_name
        ORDER BY d.year, d.month;
    """,
    "R2 - Sales by Store and Channel": """
        SELECT s.store_name, c.channel_name,
               SUM(f.net_sales) AS total_net_sales
        FROM FactSales f
        JOIN DimStore s ON f.store_key = s.store_key
        JOIN DimChannel c ON f.channel_key = c.channel_key
        GROUP BY s.store_name, c.channel_name
        ORDER BY total_net_sales DESC;
    """,
    "R3 - Top Categories and Brands": """
        SELECT p.category, p.brand,
               SUM(f.net_sales) AS total_revenue,
               SUM(f.quantity) AS total_units
        FROM FactSales f
        JOIN DimProduct p ON f.product_key = p.product_key
        GROUP BY p.category, p.brand
        ORDER BY total_revenue DESC;
    """,
    "R4 - Promotion Performance": """
        SELECT pr.promotion_name, pr.discount_pct,
               SUM(f.net_sales) AS total_sales,
               SUM(f.quantity) AS total_units,
               SUM(f.discount_amount) AS total_discount
        FROM FactSales f
        JOIN DimPromotion pr ON f.promotion_key = pr.promotion_key
        GROUP BY pr.promotion_name, pr.discount_pct
        ORDER BY total_sales DESC;
    """,
    "R5 - Gross Profit and Margin": """
        SELECT p.category, s.store_name, d.month_name,
               SUM(f.gross_profit) AS total_gross_profit,
               ROUND(SUM(f.gross_profit) / SUM(f.net_sales) * 100, 2) AS gross_margin_pct
        FROM FactSales f
        JOIN DimProduct p ON f.product_key = p.product_key
        JOIN DimStore s ON f.store_key = s.store_key
        JOIN DimDate d ON f.date_key = d.date_key
        GROUP BY p.category, s.store_name, d.month_name
        ORDER BY gross_margin_pct DESC;
    """
}

for title, sql in QUERIES.items():
    display(Markdown(f"### {title}"))
    df = pd.read_sql_query(sql, conn)
    display(df)
    print()

---
## 9. Visualizaciones

In [ ]:
DOCS_DIR = os.path.join(ROOT, 'docs')
os.makedirs(DOCS_DIR, exist_ok=True)

# Visualization 1: Monthly Net Sales Trend
df_r1 = pd.read_sql_query("""
    SELECT d.month_name, d.month, SUM(f.net_sales) AS total_net_sales
    FROM FactSales f
    JOIN DimDate d ON f.date_key = d.date_key
    GROUP BY d.month_name, d.month
    ORDER BY d.month;
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_r1['month_name'], df_r1['total_net_sales'], marker='o', linewidth=2, color='#2196F3')
ax.set_title('R1: Monthly Net Sales Trend (Jan-Jun 2026)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Net Sales (COP)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'visualization_monthly_sales.png'), dpi=150)
plt.show()
print("[OK] Saved: visualization_monthly_sales.png")

In [ ]:
# Visualization 2: Sales by Store and Channel
df_r2 = pd.read_sql_query("""
    SELECT s.store_name, c.channel_name, SUM(f.net_sales) AS total_net_sales
    FROM FactSales f
    JOIN DimStore s ON f.store_key = s.store_key
    JOIN DimChannel c ON f.channel_key = c.channel_key
    GROUP BY s.store_name, c.channel_name
    ORDER BY total_net_sales DESC;
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{row['store_name']}\n({row['channel_name']})" for _, row in df_r2.iterrows()]
bars = ax.bar(labels, df_r2['total_net_sales'], color=['#4CAF50', '#FF9800', '#E91E63'])
ax.set_title('R2: Sales Performance by Store and Channel', fontsize=14, fontweight='bold')
ax.set_ylabel('Net Sales (COP)')
ax.set_xlabel('Store / Channel')
for bar, val in zip(bars, df_r2['total_net_sales']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DOCS_DIR, 'visualization_sales_by_store.png'), dpi=150)
plt.show()
print("[OK] Saved: visualization_sales_by_store.png")

---
## 10. Resumen del Modelo Dimensional

### Tablas y Relaciones

| Dimension | Key | Registros | Descripción |
|-----------|-----|-----------|-------------|
| `DimDate` | `date_key` (PK) | 181 | Calendario Jan-Jun 2026 |
| `DimProduct` | `product_key` (PK) | 8 | 4 categorías × 2 marcas |
| `DimStore` | `store_key` (PK) | 3 | 2 tiendas físicas + 1 online |
| `DimChannel` | `channel_key` (PK) | 3 | Physical / Online |
| `DimPromotion` | `promotion_key` (PK) | 6 | Tipos de promoción |
| `FactSales` | `sale_id` (PK) | 1,000 | Una fila por línea de venta |

### Fórmulas de Medidas

| Medida | Fórmula |
|--------|---------|
| `gross_sales` | `quantity × list_price` |
| `net_sales` | `quantity × unit_price_sale` |
| `discount_amount` | `gross_sales − net_sales` |
| `cost_amount` | `quantity × unit_cost` |
| `gross_profit` | `net_sales − cost_amount` |

### Requisitos Cubiertos

| Requisito | Query | Visualización |
|-----------|-------|---------------|
| R1 - Tendencia mensual | `R1 - Monthly Net Sales Trend` | Line chart |
| R2 - Ventas por tienda | `R2 - Sales by Store and Channel` | Bar chart |
| R3 - Top categorías | `R3 - Top Categories and Brands` | - |
| R4 - Rendimiento promos | `R4 - Promotion Performance` | - |
| R5 - Margen bruto | `R5 - Gross Profit and Margin` | - |

In [ ]:
conn.close()
print("[OK] Notebook completed. Database closed.")